# BGL log anomaly detection and alerting

**Research questions**
- **RQ1:** Does a parser-free representation match or outperform Drain-template representations for semi-supervised and unsupervised anomaly detection?
- **RQ2:** How do training methods (semi-supervised vs unsupervised) and data splitting (chronological vs random) affect detection performance over the seven-month log?
- **RQ3:** Can window-level anomalies be turned into interpretable natural-language alerts that correctly identify the root-cause message?

## Imports and Root Folder

In [2]:
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import random
import re
import shutil
import time
import urllib.request
import warnings
import zipfile

from drain3 import TemplateMiner
from drain3.masking import MaskingInstruction
from drain3.template_miner_config import TemplateMinerConfig
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Repository root:", ROOT)

Repository root: /Users/shreyabharat/Projects/COS802-Project


## 0. Configuration

In [3]:
SMOKE_TEST = False        # True = quick check on the first 300k lines
REUSE_EXISTING = True     # True = skip download / parse / NLP / windows if their outputs already exist

SEED = 42

# Data: Loghub v8 on Zenodo (record 8196385), CC BY 4.0
BGL_URL = "https://zenodo.org/records/8196385/files/BGL.zip?download=1"
BGL_ZIP, BGL_LOG = "BGL.zip", "BGL.log"
EXPECTED_LINES = 4_747_963
MAX_LINES = None                   # e.g. 500_000 for a quicker partial run

# Parsing and windows
DRAIN = {"sim_th": 0.5, "depth": 4, "max_children": 100}
WINDOW_MINUTES = 5
TRAIN_FRAC, VAL_FRAC = 0.7, 0.15   

# NLP preprocessing
SPACY_MODEL = "en_core_web_sm"
MIN_TOKEN_LEN = 2
PROTECTED_WORDS = {"data", "rts", "ciod", "ddr", "tlb", "ras", "iar", "dear", "cpu", "rbs"}

# Representations (RQ1 fair pairs: tfidf vs pf_tfidf, sbert vs pf_sbert)
FEATURE_SETS = ["count", "tfidf", "sbert", "pf_tfidf", "pf_sbert"]
NOVELTY_FEATURE = True
TFIDF_MAX_FEATURES = 2000
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
ENCODER_BACKEND = "sentence-transformers"     

# Detectors and training regimes
MODELS = ["iforest", "autoencoder", "pca"]
REGIMES = ["unsupervised", "semi_supervised"]
IFOREST = {"n_estimators": 300, "max_samples": 256}
AUTOENCODER = {"hidden": [256, 64], "latent": 16, "epochs": 25, "batch_size": 256, "lr": 1e-3, "patience": 5}
PCA_VARIANCE = 0.95 # keep the components that explain 95% of the variance
SUPERVISED_REFERENCE = True # logistic regression trained WITH labels: a ceiling, never a method

SEEDS = [42, 43, 44, 45, 46]    # RQ1: seeds for isolation forest and the autoencoder
SPLIT_MODELS = ["iforest", "supervised_ref"]    # RQ2: the supervised model is the control
SPLIT_SEEDS = [42, 43, 44]
MIN_EFFECT = 0.01    # smallest AUC-PR difference treated as meaningful

ALERT_TOP_K, ALERT_EXAMPLES = 3, 25

if SMOKE_TEST:
    MAX_LINES = 300_000
    ENCODER_BACKEND = "hashing"
    AUTOENCODER["epochs"] = 5
    SEEDS, SPLIT_SEEDS = [42, 43], [42, 43]

OUT = ROOT / "smoke" if SMOKE_TEST else ROOT
RAW = ROOT / "data" / "raw"
INTERIM = OUT / "data" / "interim"
PROCESSED = OUT / "data" / "processed"
RESULTS = OUT / "results"
FIGURES = OUT / "reports" / "figures"
TABLES_DIR = OUT / "reports" / "tables"
for folder in (RAW, INTERIM, PROCESSED, RESULTS, FIGURES, TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Feature sets:", FEATURE_SETS, "| detectors:", MODELS, "| outputs:", OUT)

Feature sets: ['count', 'tfidf', 'sbert', 'pf_tfidf', 'pf_sbert'] | detectors: ['iforest', 'autoencoder', 'pca'] | outputs: /Users/shreyabharat/Projects/COS802-Project


## 1. Utility functions

In [4]:
# Prints a message with the time, showing how a long run is progressing
def log(msg: str) -> None:
    print(f"{datetime.now():%H:%M:%S} | {msg}", flush=True)

# Shows how long it took to execute a block of code
@contextmanager
def timer(msg: str):
    t0 = time.perf_counter()
    log(f"{msg} ...")
    yield
    log(f"{msg} done in {time.perf_counter() - t0:.1f}s")

# Sets the random seed for Python, Numpy, and PyTorch to the selected random seed for reproducibility
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass

# Assisting with running locally vs on Colab
def get_device() -> str:
    import torch
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

## 2. Download BGL Dataset

In [5]:
def download_bgl() -> Path:
    log_path, zip_path = RAW / BGL_LOG, RAW / BGL_ZIP
    if log_path.exists():
        log(f"{BGL_LOG} already present - skipping download")
    else:
        if not zip_path.exists():
            log(f"Downloading {BGL_URL}")
            req = urllib.request.Request(BGL_URL, headers={"User-Agent": "Mozilla/5.0 (research download)"})
            with urllib.request.urlopen(req) as resp, open(zip_path, "wb") as out:
                total = int(resp.headers.get("Content-Length", 0)) or None
                with tqdm(total=total, unit="B", unit_scale=True, desc=BGL_ZIP) as bar:
                    while chunk := resp.read(1 << 20):
                        out.write(chunk)
                        bar.update(len(chunk))
        log(f"Extracting {BGL_ZIP}")
        with zipfile.ZipFile(zip_path) as zf:
            member = next(m for m in zf.namelist() if m.endswith(BGL_LOG))
            with zf.open(member) as src, open(log_path, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 24)

    n_lines, h = 0, hashlib.sha256()
    with open(log_path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            n_lines += chunk.count(b"\n")
            h.update(chunk)
    log(f"{BGL_LOG}: {n_lines:,} lines, {log_path.stat().st_size / 1e6:,.0f} MB, sha256={h.hexdigest()[:16]}...")
    if abs(n_lines - EXPECTED_LINES) > 1:
        log(f"WARNING: expected {EXPECTED_LINES:,} lines - check the download")
    return log_path

In [6]:
# Run the download, or check if file already exists
with timer("download"):
    download_bgl()

10:19:58 | download ...
10:19:58 | BGL.log already present - skipping download
10:19:59 | BGL.log: 4,747,963 lines, 743 MB, sha256=666130b15ef44eb3...
10:19:59 | download done in 0.7s


## 3. Parsing (Drain3 template vs parser-free view)

In [ ]:
FIELDS = ["label", "ts", "date", "node", "time", "node_repeat", "type", "component", "level", "content"]
MASKS = [
    (r"R\d+-M\d+(?:-[A-Z0-9]+)*(?::J\d+-U\d+)?", "NODE"),
    (r"\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b", "IP"),
    (r"\b0x[0-9a-fA-F]+\b", "HEX"),
    (r"\b[0-9a-fA-F]{8,}\b", "HEX"),
    (r"(?:/[\w.\-]+){2,}", "PATH"),
    (r"(?<![A-Za-z_])[-+]?\d+(?:\.\d+)?(?![A-Za-z_])", "NUM"),
]
COMPILED_MASKS = [(re.compile(pattern), f"<{name}>") for pattern, name in MASKS]

def mask_text(content: str) -> str:
    """Parser-free view: apply the regex masks and nothing else."""
    for rx, placeholder in COMPILED_MASKS:
        content = rx.sub(placeholder, content)
    return re.sub(r"\s+", " ", content).strip()

def split_line(line: str) -> dict | None:
    """Split one raw BGL line into its fields. Returns None for malformed lines."""
    parts = line.rstrip("\n").split(maxsplit=9)
    if len(parts) < 9:
        return None
    if len(parts) == 9: # a line with an empty message
        parts.append("")
    rec = dict(zip(FIELDS, parts))
    try:
        rec["ts"] = int(rec["ts"])
    except ValueError:
        return None
    return rec

def make_miner() -> TemplateMiner:
    """A Drain3 template miner that uses the same masks as the parser-free view."""
    c = TemplateMinerConfig()
    c.drain_sim_th, c.drain_depth, c.drain_max_children = DRAIN["sim_th"], DRAIN["depth"], DRAIN["max_children"]
    c.masking_instructions = [MaskingInstruction(pattern, name) for pattern, name in MASKS]
    c.profiling_enabled = False
    return TemplateMiner(config=c)

In [8]:
# parse_bgl() reads the log once, line by line, and for each line:
# 1. splits the fields
# 2. passes the message to Drain3 and records the template id (cluster_id)
# 3. masks the message and gives each distinct masked text an id (msg_id)
# 4. writes the results to data/interim/bgl_parsed.parquet 

def parse_bgl() -> None:
    miner = make_miner()
    schema = pa.schema([
        ("line_id", pa.int64()), ("label", pa.string()), ("is_anomaly", pa.bool_()), ("ts", pa.int64()),
        ("node", pa.string()), ("component", pa.string()), ("level", pa.string()),
        ("content", pa.string()), ("cluster_id", pa.int32()), ("msg_id", pa.int32()),
    ])
    writer = pq.ParquetWriter(INTERIM / "bgl_parsed.parquet", schema, compression="zstd")
    rows, msg_ids, msg_counts, n_bad = [], {}, [], 0
    t_start = time.perf_counter()

    with open(RAW / BGL_LOG, encoding="utf-8", errors="replace") as fh:
        for i, line in enumerate(tqdm(fh, total=MAX_LINES or EXPECTED_LINES, desc="parsing", mininterval=5)):
            if MAX_LINES and i >= MAX_LINES:
                break
            rec = split_line(line)
            if rec is None:
                n_bad += 1
                continue
            cluster_id = miner.add_log_message(rec["content"])["cluster_id"]     # template-based view
            msg_id = msg_ids.setdefault(mask_text(rec["content"]), len(msg_ids))  # parser-free view
            if msg_id == len(msg_counts):
                msg_counts.append(0)
            msg_counts[msg_id] += 1
            rows.append({"line_id": i, "label": rec["label"], "is_anomaly": rec["label"] != "-", "ts": rec["ts"],
                         "node": rec["node"], "component": rec["component"], "level": rec["level"],
                         "content": rec["content"], "cluster_id": cluster_id, "msg_id": msg_id})
            if len(rows) >= 200_000:
                writer.write_table(pa.Table.from_pylist(rows, schema=schema))
                rows.clear()
    if rows:
        writer.write_table(pa.Table.from_pylist(rows, schema=schema))
    writer.close()

    templates = pd.DataFrame([{"cluster_id": c.cluster_id, "template": c.get_template(), "size": c.size}
                              for c in miner.drain.clusters]).sort_values("cluster_id")
    templates.to_csv(INTERIM / "templates.csv", index=False)
    messages = pd.DataFrame({"msg_id": range(len(msg_ids)), "template": list(msg_ids), "size": msg_counts})
    messages.to_csv(INTERIM / "messages.csv", index=False)

    stats = {"lines": int(sum(msg_counts)), "malformed": n_bad, "templates": len(templates),
             "masked_messages": len(messages), "wall_seconds": round(time.perf_counter() - t_start, 1)}
    (INTERIM / "parse_stats.json").write_text(json.dumps(stats, indent=2))
    log(f"Parsed {stats['lines']:,} lines ({n_bad} malformed skipped): {len(templates)} templates, "
        f"{len(messages):,} masked messages")

In [9]:
if REUSE_EXISTING and (INTERIM / "bgl_parsed.parquet").exists():
    print("Parsed data found - skipping. Set REUSE_EXISTING = False to parse again.")
else:
    with timer("parse (Drain3 + masking)"):
        parse_bgl()
print(json.dumps(json.loads((INTERIM / "parse_stats.json").read_text()), indent=1))

10:54:04 | parse (Drain3 + masking) ...


parsing: 100%|██████████| 4747963/4747963 [01:04<00:00, 73230.52it/s]

10:55:09 | Parsed 4,747,963 lines (0 malformed skipped): 392 templates, 16,107 masked messages
10:55:09 | parse (Drain3 + masking) done in 65.0s
{
 "lines": 4747963,
 "malformed": 0,
 "templates": 392,
 "masked_messages": 16107,
 "wall_seconds": 65.0
}


## 4. NLP preprocessing

In [10]:
MASK_TOKEN = re.compile(r"<[A-Z*]+>")   # <*>, <NUM>, <HEX>, <NODE>, ...
CAMEL = re.compile(r"(?<=[a-z])(?=[A-Z])")  # position between 'ciod' and 'Startup'
NON_ALPHA = re.compile(r"[^A-Za-z]+")

def clean_text(template: str) -> str:
    t = MASK_TOKEN.sub(" ", template)
    t = CAMEL.sub(" ", t)
    t = re.sub(r"[_:=\[\](){},;]+", " ", t)
    t = re.sub(r"\.{2,}", " ", t)   # 'enabled.....' -> 'enabled'
    return re.sub(r"\s+", " ", t).strip(" .")

def tokenize(template: str) -> list[str]:
    t = MASK_TOKEN.sub(" ", template)
    t = CAMEL.sub(" ", t)
    return [tok.lower() for tok in NON_ALPHA.split(t) if tok]

def preprocess(table: pd.DataFrame) -> pd.DataFrame:
    import spacy
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
    table = table.copy()
    table["clean"] = table["template"].map(clean_text)
    table["tokens"] = table["template"].map(tokenize)
    table["lemmas"] = [
        [tok.lower_ if tok.lower_ in PROTECTED_WORDS else tok.lemma_.lower() for tok in doc
         if not tok.is_stop and tok.is_alpha and len(tok) >= MIN_TOKEN_LEN]
        for doc in nlp.pipe((" ".join(toks) for toks in table["tokens"]), batch_size=256)
    ]
    vocab_tok = len({t for ts in table["tokens"] for t in ts})
    vocab_lem = len({t for ts in table["lemmas"] for t in ts})
    log(f"{len(table):,} rows | vocabulary {vocab_tok} tokens -> {vocab_lem} lemmas")
    return table

def save_table(table: pd.DataFrame, path: Path) -> None:
    out = table.copy()
    out["tokens"], out["lemmas"] = out["tokens"].map(json.dumps), out["lemmas"].map(json.dumps)
    out.to_csv(path, index=False)

def load_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, keep_default_na=False)
    df["tokens"], df["lemmas"] = df["tokens"].map(json.loads), df["lemmas"].map(json.loads)
    return df

In [11]:
if REUSE_EXISTING and (INTERIM / "messages_nlp.csv").exists():
    print("NLP outputs found - skipping.")
else:
    with timer("NLP preprocessing"):
        for name in ("templates", "messages"):
            save_table(preprocess(pd.read_csv(INTERIM / f"{name}.csv", keep_default_na=False)),
                       INTERIM / f"{name}_nlp.csv")

11:05:58 | NLP preprocessing ...
11:06:03 | 392 rows | vocabulary 638 tokens -> 506 lemmas
11:06:24 | 16,107 rows | vocabulary 818 tokens -> 672 lemmas
11:06:24 | NLP preprocessing done in 26.5s


## 5. Time windows

In [12]:
SEVERE_LEVELS = {"FATAL", "FAILURE", "SEVERE", "ERROR"}
UNIT_KEYS = {"template": "cluster_id", "message": "msg_id"}   # the id column of each view


def load_lines(columns: list[str]) -> pd.DataFrame:
    """Parsed lines in time order (ties broken by original line number)."""
    df = pd.read_parquet(INTERIM / "bgl_parsed.parquet", columns=["line_id", *columns])
    return df.sort_values(["ts", "line_id"], kind="stable").reset_index(drop=True)


def build_windows(lines: pd.DataFrame, tables: dict) -> tuple[pd.DataFrame, dict]:
    size = WINDOW_MINUTES * 60
    ts = lines["ts"].to_numpy()
    t0 = int(ts.min())
    win = (ts - t0) // size # window number of every line
    n_windows = int(win.max()) + 1

    counts = {}
    for unit, key in UNIT_KEYS.items():
        col_of = pd.Series(np.arange(len(tables[unit])), index=tables[unit][key].values)
        cols = col_of.reindex(lines[key].values).to_numpy()
        assert not np.isnan(cols).any(), f"some {key} values are missing from the {unit} table"
        counts[unit] = sp.csr_matrix((np.ones(len(lines), dtype=np.float32), (win, cols.astype(np.int64))),
                                     shape=(n_windows, len(tables[unit])))     # duplicates are summed

    per_window = lambda values: np.bincount(win, weights=values, minlength=n_windows)
    node_codes = pd.factorize(lines["node"])[0].astype(np.int64)
    pairs = np.unique(win * (node_codes.max() + 1) + node_codes)    # distinct (window, node) pairs
    meta = pd.DataFrame({
        "win_id": np.arange(n_windows),
        "start_ts": t0 + np.arange(n_windows) * size,
        "n_lines": per_window(np.ones(len(lines))).astype(int),
        "n_anom_lines": per_window(lines["is_anomaly"].to_numpy().astype(float)).astype(int),
        "n_severe_lines": per_window(lines["level"].isin(SEVERE_LEVELS).to_numpy().astype(float)).astype(int),
        "n_nodes": np.bincount(pairs // (node_codes.max() + 1), minlength=n_windows).astype(int),
    })
    meta["end_ts"] = meta["start_ts"] + size
    meta["label"] = (meta["n_anom_lines"] > 0).astype(int)

    keep = meta["n_lines"].to_numpy() >= 1  # drop empty windows
    meta = meta[keep].reset_index(drop=True)
    counts = {unit: m[np.flatnonzero(keep)].tocsr() for unit, m in counts.items()}
    log(f"{len(meta):,} windows kept ({(~keep).sum():,} empty dropped); anomalous windows {meta['label'].mean():.1%}; "
        f"line-level anomaly rate {lines['is_anomaly'].mean():.2%}")
    return meta, counts


def assign_split(n: int, mode: str = "chronological", seed: int = SEED) -> np.ndarray:
    """'train' / 'val' / 'test' for each of n windows (which are in time order)."""
    order = np.arange(n) if mode == "chronological" else np.random.default_rng(seed).permutation(n)
    n_train, n_val = int(n * TRAIN_FRAC), int(n * VAL_FRAC)
    split = np.empty(n, dtype=object)
    split[order[:n_train]] = "train"
    split[order[n_train:n_train + n_val]] = "val"
    split[order[n_train + n_val:]] = "test"
    return split

In [13]:
if REUSE_EXISTING and (PROCESSED / "windows.parquet").exists():
    print("Windows found - skipping. Set REUSE_EXISTING = False to rebuild.")
else:
    with timer("windowing"):
        _tables = {"template": load_table(INTERIM / "templates_nlp.csv"),
                   "message": load_table(INTERIM / "messages_nlp.csv")}
        _meta, _counts = build_windows(load_lines(["ts", "cluster_id", "msg_id", "is_anomaly", "level", "node"]), _tables)
        _meta.to_parquet(PROCESSED / "windows.parquet", index=False)
        for unit, m in _counts.items():
            sp.save_npz(PROCESSED / f"W_{unit}.npz", m)

20:56:55 | windowing ...
20:56:56 | 14,501 windows kept (47,339 empty dropped); anomalous windows 7.3%; line-level anomaly rate 7.34%
20:56:56 | windowing done in 1.6s
